In [1]:
import os
import math
import json
import numpy as np
import pandas as pd
import seaborn as sns
from collections import Counter
import matplotlib.colors
import matplotlib.cm as cm
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from datetime import datetime
# from patsy import dmatrices
from patsy import dmatrix
from scipy.stats import ttest_rel
from scipy.stats import ttest_ind
import statsmodels.formula.api as smf
main_path = r'/home/20250114zmz_kd/'

In [2]:
from causalinference import CausalModel
from causalinference.utils import random_data

In [3]:
CIs = {'90': 1.645, '95': 1.96, '99': 2.576}

In [4]:
labels = ['US', 'Others']

In [5]:
data = r'GraduationPaper/RevisetoJournal/9991-MergedData_similarity.csv'
d = pd.read_csv(main_path + data)
del d['Unnamed: 0']
print(d .shape)
d .columns

(317275, 102)


Index(['work_id', 'PublishedYear', 'Facility', 'num_fac', 'paper_type',
       'paper_language', 'novel_uzzi', 'novel_uzzi_bin', 'num_fac_scientist',
       'ratio_fac_scientist',
       ...
       'ab_length', 'mean_career_age', 'ex_ld_avg_avgimpact',
       'ex_ld_avg_insthindex', 'ex_ld_avg_before_year_prod_fac',
       'ex_ld_avg_before_year_with_ih', 'ex_ld_avg_before_year_co_lead',
       'ex_ld_avg_before_year_participation', 'knowledge_proximity_mean',
       'knowledge_proximity_max'],
      dtype='object', length=102)

In [6]:
d['CoType'].unique()

array(['Collaboration', 'Service', 'Participation'], dtype=object)

In [7]:
d = d[d['CoType'].isin(['Service','Participation'])]
print(d .shape)

(260260, 102)


In [8]:
d['reg_class_bin'] = d.apply(lambda row: 1 if row['CoType'] == 'Participation' else 0, axis = 1)
d['reg_class_bin'].value_counts()

reg_class_bin
0    232552
1     27708
Name: count, dtype: int64

In [9]:
# 假设 df 是你的 dataframe
d['PublishedYear'] = d['PublishedYear'].astype('category')

In [10]:
d .columns

Index(['work_id', 'PublishedYear', 'Facility', 'num_fac', 'paper_type',
       'paper_language', 'novel_uzzi', 'novel_uzzi_bin', 'num_fac_scientist',
       'ratio_fac_scientist',
       ...
       'mean_career_age', 'ex_ld_avg_avgimpact', 'ex_ld_avg_insthindex',
       'ex_ld_avg_before_year_prod_fac', 'ex_ld_avg_before_year_with_ih',
       'ex_ld_avg_before_year_co_lead', 'ex_ld_avg_before_year_participation',
       'knowledge_proximity_mean', 'knowledge_proximity_max', 'reg_class_bin'],
      dtype='object', length=103)

In [11]:
co_feats_ = ["lnnum_author", "international", "lnnum_reference", "num_fac", "SDG",
             "lnmean_career_age", "lnex_ld_avg_avgimpact", "lnex_ld_avg_insthindex",
             "ex_ld_bin_gs", "ex_ld_bin_sameC", "knowledge_proximity_mean",
             "lnex_ld_avg_before_year_prod_fac", "ex_ld_max_before_year_with_ih_bin",
             'Agricultural and Biological Sciences',
             'Arts and Humanities', 'Biochemistry, Genetics and Molecular Biology',
             'Business, Management and Accounting', 'Chemical Engineering',
             'Chemistry', 'Computer Science', 'Decision Sciences', 'Dentistry',
             'Earth and Planetary Sciences', 'Economics, Econometrics and Finance',
             'Energy', 'Engineering', 'Environmental Science', 'Health Professions',
             'Immunology and Microbiology', 'Materials Science', 'Mathematics',
             'Medicine', 'Neuroscience', 'Nursing','Pharmacology, Toxicology and Pharmaceutics', 'Physics and Astronomy',
             'Psychology', 'Social Sciences', 'Veterinary',]

In [12]:
# 1. 把 DataFrame 里的空格替换成下划线
d.columns = d.columns.str.replace(' ', '_')
d.columns = d.columns.str.replace(',', '')
print(d .columns)
# 2. 把特征列表里的空格也替换掉
co_feats_ = [f.replace(' ', '_') for f in co_feats_]
co_feats_ = [f.replace(',', '') for f in co_feats_]
print(co_feats_)

Index(['work_id', 'PublishedYear', 'Facility', 'num_fac', 'paper_type',
       'paper_language', 'novel_uzzi', 'novel_uzzi_bin', 'num_fac_scientist',
       'ratio_fac_scientist',
       ...
       'mean_career_age', 'ex_ld_avg_avgimpact', 'ex_ld_avg_insthindex',
       'ex_ld_avg_before_year_prod_fac', 'ex_ld_avg_before_year_with_ih',
       'ex_ld_avg_before_year_co_lead', 'ex_ld_avg_before_year_participation',
       'knowledge_proximity_mean', 'knowledge_proximity_max', 'reg_class_bin'],
      dtype='object', length=103)
['lnnum_author', 'international', 'lnnum_reference', 'num_fac', 'SDG', 'lnmean_career_age', 'lnex_ld_avg_avgimpact', 'lnex_ld_avg_insthindex', 'ex_ld_bin_gs', 'ex_ld_bin_sameC', 'knowledge_proximity_mean', 'lnex_ld_avg_before_year_prod_fac', 'ex_ld_max_before_year_with_ih_bin', 'Agricultural_and_Biological_Sciences', 'Arts_and_Humanities', 'Biochemistry_Genetics_and_Molecular_Biology', 'Business_Management_and_Accounting', 'Chemical_Engineering', 'Chemistry', 'Comp

In [13]:
X = dmatrix(formula_like=' + '.join(co_feats_), data=d, return_type="dataframe")

In [14]:
list(X.columns)

['Intercept',
 'international[T.international]',
 'SDG[T.True]',
 'ex_ld_bin_gs[T.GlobalSouth]',
 'ex_ld_bin_sameC[T.Same]',
 'ex_ld_max_before_year_with_ih_bin[T.True]',
 'lnnum_author',
 'lnnum_reference',
 'num_fac',
 'lnmean_career_age',
 'lnex_ld_avg_avgimpact',
 'lnex_ld_avg_insthindex',
 'knowledge_proximity_mean',
 'lnex_ld_avg_before_year_prod_fac',
 'Agricultural_and_Biological_Sciences',
 'Arts_and_Humanities',
 'Biochemistry_Genetics_and_Molecular_Biology',
 'Business_Management_and_Accounting',
 'Chemical_Engineering',
 'Chemistry',
 'Computer_Science',
 'Decision_Sciences',
 'Dentistry',
 'Earth_and_Planetary_Sciences',
 'Economics_Econometrics_and_Finance',
 'Energy',
 'Engineering',
 'Environmental_Science',
 'Health_Professions',
 'Immunology_and_Microbiology',
 'Materials_Science',
 'Mathematics',
 'Medicine',
 'Neuroscience',
 'Nursing',
 'Pharmacology_Toxicology_and_Pharmaceutics',
 'Physics_and_Astronomy',
 'Psychology',
 'Social_Sciences',
 'Veterinary']

In [15]:
co_feats = [
 'international[T.international]',
 'SDG[T.True]',
 'ex_ld_bin_gs[T.GlobalSouth]',
 'ex_ld_bin_sameC[T.Same]',
 'ex_ld_max_before_year_with_ih_bin[T.True]',
 'lnnum_author',
 'lnnum_reference',
 'num_fac',
 'lnmean_career_age',
 'lnex_ld_avg_avgimpact',
 'lnex_ld_avg_insthindex',
 'knowledge_proximity_mean',
 'lnex_ld_avg_before_year_prod_fac',
 'Agricultural_and_Biological_Sciences',
 'Arts_and_Humanities',
 'Biochemistry_Genetics_and_Molecular_Biology',
 'Business_Management_and_Accounting',
 'Chemical_Engineering',
 'Chemistry',
 'Computer_Science',
 'Decision_Sciences',
 'Dentistry',
 'Earth_and_Planetary_Sciences',
 'Economics_Econometrics_and_Finance',
 'Energy',
 'Engineering',
 'Environmental_Science',
 'Health_Professions',
 'Immunology_and_Microbiology',
 'Materials_Science',
 'Mathematics',
 'Medicine',
 'Neuroscience',
 'Nursing',
 'Pharmacology_Toxicology_and_Pharmaceutics',
 'Physics_and_Astronomy',
 'Psychology',
 'Social_Sciences',
 'Veterinary'
]

In [16]:
X['CoType'] = d['CoType']
X['reg_class_bin'] = d['reg_class_bin']
X['novel_uzzi_bin'] = d['novel_uzzi_bin']

In [17]:
# Y is the outcome, D is treatment status, and X is the independent variable
causal = CausalModel(Y=X['novel_uzzi_bin'].values, D=X['reg_class_bin'].values, \
                     X=X[co_feats].values)

In [18]:
print(causal.summary_stats)


Summary Statistics

                    Controls (N_c=232552)       Treated (N_t=27708)             
       Variable         Mean         S.d.         Mean         S.d.     Raw-diff
--------------------------------------------------------------------------------
              Y        0.373        0.484        0.341        0.474       -0.032

                    Controls (N_c=232552)       Treated (N_t=27708)             
       Variable         Mean         S.d.         Mean         S.d.     Nor-diff
--------------------------------------------------------------------------------
             X0        0.439        0.496        0.806        0.396        0.818
             X1        0.486        0.500        0.433        0.495       -0.106
             X2        0.063        0.243        0.081        0.272        0.068
             X3        0.531        0.499        0.257        0.437       -0.585
             X4        0.681        0.466        0.933        0.250        0.675
      

In [19]:
causal.est_propensity()

In [20]:
# Propensity model results
print(causal.propensity)


Estimated Parameters of Propensity Score

                    Coef.       S.e.          z      P>|z|      [95% Conf. int.]
--------------------------------------------------------------------------------
     Intercept     -4.430      0.124    -35.739      0.000     -4.673     -4.187
            X0      0.864      0.018     48.022      0.000      0.829      0.899
            X1     -0.109      0.014     -7.518      0.000     -0.137     -0.080
            X2     -0.317      0.026    -12.044      0.000     -0.369     -0.266
            X3     -0.617      0.017    -36.380      0.000     -0.650     -0.584
            X4      1.508      0.028     54.438      0.000      1.454      1.563
            X5      0.872      0.012     70.565      0.000      0.848      0.896
            X6      0.219      0.014     15.422      0.000      0.191      0.246
            X7      0.621      0.010     60.354      0.000      0.600      0.641
            X8      0.052      0.025      2.044      0.041      0.

In [21]:
causal.propensity['fitted']

array([0.12111662, 0.03076903, 0.11167684, ..., 0.0331173 , 0.03303182,
       0.30494011])

In [22]:
d['CoType'].unique()

array(['Service', 'Participation'], dtype=object)

In [23]:
for feat in co_feats:
    print(feat)
    af_avg = np.mean(X.loc[X['CoType']=='Participation', feat])
    nam_avg = np.mean(X.loc[X['CoType']=='Service', feat])
    print('\tParticipation:\t', af_avg)
    print('\tService:\t', nam_avg)
    print('\tDiff:\t', af_avg-nam_avg)
    print('\tT-test:\t', ttest_ind(X.loc[X['CoType']=='Participation', feat], X.loc[X['CoType']=='Service', feat])[1])

international[T.international]
	Participation:	 0.8057239786343294
	Service:	 0.43872338229729263
	Diff:	 0.3670005963370368
	T-test:	 0.0
SDG[T.True]
	Participation:	 0.43279919156922186
	Service:	 0.485650521173759
	Diff:	 -0.052851329604537145
	T-test:	 3.043466156802714e-62
ex_ld_bin_gs[T.GlobalSouth]
	Participation:	 0.08055435253356431
	Service:	 0.06303536413361313
	Diff:	 0.017518988399951183
	T-test:	 4.51797757440686e-29
ex_ld_bin_sameC[T.Same]
	Participation:	 0.2568933160098167
	Service:	 0.5312059238363893
	Diff:	 -0.27431260782657263
	T-test:	 0.0
ex_ld_max_before_year_with_ih_bin[T.True]
	Participation:	 0.9331961888263317
	Service:	 0.6808369740961161
	Diff:	 0.2523592147302156
	T-test:	 0.0
lnnum_author
	Participation:	 2.5703475962738844
	Service:	 1.9980434311474133
	Diff:	 0.5723041651264711
	T-test:	 0.0
lnnum_reference
	Participation:	 3.7532892508659788
	Service:	 3.6534719171420424
	Diff:	 0.09981733372393631
	T-test:	 3.617455834299364e-167
num_fac
	Participati

In [24]:
len(causal.propensity['fitted'])

260260

In [25]:
X['pscore'] = causal.propensity['fitted']

In [26]:
tem = X[['CoType', 'pscore']].sort_values(by = ['pscore'])
tem['index'] = tem.index

In [27]:
tem = tem.values.tolist()

In [28]:
tem[-10:]

[['Service', 0.9850774971640465, 135802],
 ['Participation', 0.9927400435568267, 181100],
 ['Participation', 0.9957372765965338, 280659],
 ['Participation', 0.9957446323367778, 280657],
 ['Participation', 0.9957643778683262, 280656],
 ['Participation', 0.9957794711115842, 280658],
 ['Participation', 0.99769832485966, 280654],
 ['Participation', 0.9976994175059484, 280660],
 ['Participation', 0.9977002739420014, 280655],
 ['Participation', 0.9977006553604868, 280653]]

In [29]:
leng = len(tem)

In [30]:
print(leng)

260260


In [31]:
pairs = {}
for i, elms in enumerate(tem):
    gen, score, ix = elms
    if gen == 'Participation':
        pre_nam_ix, nex_nam_ix = 0, 0
        j = i-1
        while j >= 0:
            if tem[j][0] != 'Service':
                j -= 1
            else:
                break
        if j >= 0:
            pre_nam_ix = j
        n = i+1
        while n <= leng-1:
            if tem[n][0] != 'Service':
                n += 1
            else:
                break
        if n <= leng-1:
            nex_nam_ix = n
        if abs(score - tem[pre_nam_ix][1]) <= abs(score - tem[nex_nam_ix][1]):
            pairs[ix] = tem[pre_nam_ix][2]
        else:
            pairs[ix] = tem[nex_nam_ix][2]

In [32]:
len(pairs)

27708

In [33]:
for feat in co_feats:
    print('\n')
    print('Feat:', feat, '\n')
    print('Before matching:\n')
    af_avg = np.mean(X.loc[X['CoType']=='Participation', feat])
    nam_avg = np.mean(X.loc[X['CoType']=='Service', feat])
    print('\tParticipation:\t', af_avg)
    print('\tService:\t', nam_avg)
    print('\tDiff:\t', af_avg-nam_avg)
    print('\tT-test:\t', ttest_ind(X.loc[X['CoType']=='Participation', feat], X.loc[X['CoType']=='Service', feat])[1])

    print('\nAfter matching:\n')
    af_avg = np.mean(X.loc[pairs.keys(), feat])
    nam_avg = np.mean(X.loc[pairs.values(), feat])
    print('\tParticipation:\t', af_avg)
    print('\tService:\t', nam_avg)
    print('\tDiff:\t', af_avg-nam_avg)
    print('\tT-test:\t', ttest_rel(X.loc[pairs.keys(), feat], X.loc[pairs.values(), feat])[1])



Feat: international[T.international] 

Before matching:

	Participation:	 0.8057239786343294
	Service:	 0.43872338229729263
	Diff:	 0.3670005963370368
	T-test:	 0.0

After matching:

	Participation:	 0.8057239786343294
	Service:	 0.8074563303017179
	Diff:	 -0.0017323516673884987
	T-test:	 0.49509562019863673


Feat: SDG[T.True] 

Before matching:

	Participation:	 0.43279919156922186
	Service:	 0.485650521173759
	Diff:	 -0.052851329604537145
	T-test:	 3.043466156802714e-62

After matching:

	Participation:	 0.43279919156922186
	Service:	 0.43676916414032046
	Diff:	 -0.003969972571098601
	T-test:	 0.3437501673791119


Feat: ex_ld_bin_gs[T.GlobalSouth] 

Before matching:

	Participation:	 0.08055435253356431
	Service:	 0.06303536413361313
	Diff:	 0.017518988399951183
	T-test:	 4.51797757440686e-29

After matching:

	Participation:	 0.08055435253356431
	Service:	 0.07727010249747365
	Diff:	 0.0032842500360906607
	T-test:	 0.15157440314783327


Feat: ex_ld_bin_sameC[T.Same] 

Before matc

In [34]:
for feat in co_feats:
    af_avg = np.mean(X.loc[X['CoType']=='Participation', feat])
    nam_avg = np.mean(X.loc[X['CoType']=='Service', feat])
    af_avg_ = np.mean(X.loc[pairs.keys(), feat])
    nam_avg_ = np.mean(X.loc[pairs.values(), feat])
    print(feat, ' & ', '{:6.2f}'.format(af_avg), ' & ', '{:6.2f}'.format(nam_avg), ' & ', '{:6.2f}'.format(af_avg_), ' & ', '{:6.2f}'.format(nam_avg_), ' \\\\ \hline')

international[T.international]  &    0.81  &    0.44  &    0.81  &    0.81  \\ \hline
SDG[T.True]  &    0.43  &    0.49  &    0.43  &    0.44  \\ \hline
ex_ld_bin_gs[T.GlobalSouth]  &    0.08  &    0.06  &    0.08  &    0.08  \\ \hline
ex_ld_bin_sameC[T.Same]  &    0.26  &    0.53  &    0.26  &    0.26  \\ \hline
ex_ld_max_before_year_with_ih_bin[T.True]  &    0.93  &    0.68  &    0.93  &    0.93  \\ \hline
lnnum_author  &    2.57  &    2.00  &    2.57  &    2.57  \\ \hline
lnnum_reference  &    3.75  &    3.65  &    3.75  &    3.75  \\ \hline
num_fac  &    1.56  &    1.24  &    1.56  &    1.52  \\ \hline
lnmean_career_age  &    3.11  &    3.09  &    3.11  &    3.11  \\ \hline
lnex_ld_avg_avgimpact  &    3.06  &    3.00  &    3.06  &    3.05  \\ \hline
lnex_ld_avg_insthindex  &    5.97  &    6.10  &    5.97  &    5.95  \\ \hline
knowledge_proximity_mean  &    0.77  &    0.76  &    0.77  &    0.77  \\ \hline
lnex_ld_avg_before_year_prod_fac  &    2.48  &    2.19  &    2.48  &    2.42  

In [35]:
# AS avg
np.mean(X.loc[pairs.keys(), 'novel_uzzi_bin'])

np.float64(0.3413454597950051)

In [36]:
# NAM avg
np.mean(X.loc[pairs.values(), 'novel_uzzi_bin'])

np.float64(0.3348852317020355)

In [37]:
ttest_rel(X.loc[pairs.keys(), 'novel_uzzi_bin'].apply(lambda x: 1 if x == True else 0), \
          X.loc[pairs.values(), 'novel_uzzi_bin'].apply(lambda x: 1 if x == True else 0))

TtestResult(statistic=np.float64(1.6189793195031235), pvalue=np.float64(0.10546309449482642), df=np.int64(27707))

In [38]:
y_treated = X.loc[list(pairs.keys()), 'novel_uzzi_bin'].values
y_control = X.loc[list(pairs.values()), 'novel_uzzi_bin'].values

n_boot = 1000  # bootstrap次数
att_boot = np.zeros(n_boot)
n = len(y_treated)

for i in range(n_boot):
    idx = np.random.randint(0, n, n)  # 随机抽样索引，有放回
    att_boot[i] = np.mean(y_treated[idx] - y_control[idx])

# ATT估计值
att = np.mean(y_treated - y_control)

# 95%置信区间
ci_lower = np.percentile(att_boot, 2.5)
ci_upper = np.percentile(att_boot, 97.5)

print("ATT:", att)
print("95% CI: [{:.4f}, {:.4f}]".format(ci_lower, ci_upper))

ATT: 0.006460228092969539
95% CI: [-0.0017, 0.0144]


In [39]:
d['novel_uzzi_bin'] = d['novel_uzzi_bin'].astype('int32')

In [40]:
print(d .shape)

(260260, 103)


In [41]:
dpsm = d.loc[list(pairs.keys()) + list(pairs.values())].sample(frac=1)
print(dpsm .shape)

(55416, 103)


In [42]:
dpsm .columns

Index(['work_id', 'PublishedYear', 'Facility', 'num_fac', 'paper_type',
       'paper_language', 'novel_uzzi', 'novel_uzzi_bin', 'num_fac_scientist',
       'ratio_fac_scientist',
       ...
       'mean_career_age', 'ex_ld_avg_avgimpact', 'ex_ld_avg_insthindex',
       'ex_ld_avg_before_year_prod_fac', 'ex_ld_avg_before_year_with_ih',
       'ex_ld_avg_before_year_co_lead', 'ex_ld_avg_before_year_participation',
       'knowledge_proximity_mean', 'knowledge_proximity_max', 'reg_class_bin'],
      dtype='object', length=103)

In [43]:
dpsm['work_id'].nunique()

41165

In [52]:
Update

NameError: name 'Update' is not defined

In [44]:
dpsm .to_csv(main_path + r'science_media_coverage/260128Revision/PSM-FirstCorresponding/PSM-sample-260311-africa.csv')